Import Libraries

In [1]:
import pandas as pd
import numpy as np

Load the Datasets

In [2]:
patient_df = pd.read_csv("Patient_Data.csv")
billing_df = pd.read_csv("Billing_Data.csv")

In [3]:
patient_df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [4]:
billing_df.head()

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


Show Dataset Information

In [5]:
patient_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


Select Billing Related Columns

In [6]:
billing_columns = patient_df[['PatientID',
                              'Department',
                              'Doctor',
                              'BillAmount']]

In [7]:
billing_columns.head()

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


Drop Administrative Columns

In [8]:
patient_clean = patient_df.drop(
    columns=['ReceptionistID', 'CheckInTime'],
    errors='ignore'
)

In [9]:
patient_clean.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


Find Missing Values

In [10]:
patient_clean.isnull().sum()

PatientID     0
Name          0
Department    0
Doctor        0
BillAmount    2
dtype: int64

Fill Missing BillAmount with Mean

In [12]:
mean_bill = patient_clean['BillAmount'].mean()

In [13]:
patient_clean['BillAmount'] = patient_clean['BillAmount'].fillna(mean_bill)

In [14]:
patient_clean['BillAmount'].isnull().sum()

0

Find Total Bill Amount Per Department

In [16]:
department_bill = patient_clean.groupby(
    'Department'
)['BillAmount'].sum()

In [17]:
print(department_bill)

Department
Cardiology     16200.0
Dermatology     5925.0
Neurology       5925.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


Remove Duplicate Patients

Check duplicates

In [18]:
patient_clean.duplicated(
    subset='PatientID'
).sum()

1

Remove duplicates:

In [19]:
patient_clean = patient_clean.drop_duplicates(
    subset='PatientID'
)

In [20]:
patient_clean.shape

(5, 5)

Merge Billing Dataset with Patient Dataset

In [21]:
merged_df = pd.merge(
    patient_clean,
    billing_df,
    on='PatientID',
    how='inner'
)

In [22]:
merged_df.head()

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,2000,3000
1,102,Bob,Neurology,Dr. John,5925.0,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.0,3000,3200
4,105,Eva,Dermatology,Dr. Rose,5925.0,1000,4000


Create New Patient DataFrame

In [23]:
new_patients = pd.DataFrame({
    'PatientID':[201,202],
    'Department':['Cardiology','Neurology'],
    'Doctor':['Dr. Kumar','Dr. Rao'],
    'BillAmount':[8000,9500]
})

In [24]:
new_patients

,PatientID,Department,Doctor,BillAmount
0,201,Cardiology,Dr. Kumar,8000
1,202,Neurology,Dr. Rao,9500


Concatenate New Patients (Row-wise)

In [25]:
updated_patients = pd.concat(
    [patient_clean, new_patients],
    ignore_index=True
)

In [26]:
updated_patients.tail()

,PatientID,Name,Department,Doctor,BillAmount
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,5925.0
5,201,NaN,Cardiology,Dr. Kumar,8000.0
6,202,NaN,Neurology,Dr. Rao,9500.0


Create New Billing Columns

In [28]:
new_billing_columns = pd.DataFrame({
    'InsuranceCovered':[2000,3000,2500,4000],
    'FinalAmount':[6000,7000,5500,8000]
})

Concatenate Billing Columns (Column-wise)

In [29]:
billing_extended = pd.concat(
    [billing_df, new_billing_columns],
    axis=1
)

In [30]:
billing_extended.head()

,PatientID,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,2000,3000,2000.0,6000.0
1,102,1500,3500,3000.0,7000.0
2,103,2500,5000,2500.0,5500.0
3,104,3000,3200,4000.0,8000.0
4,105,1000,4000,NaN,NaN


Final Dataset

In [31]:
final_dataset = pd.merge(
    updated_patients,
    billing_extended,
    on='PatientID',
    how='left'
)

In [32]:
final_dataset.head()

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,2000.0,3000.0,2000.0,6000.0
1,102,Bob,Neurology,Dr. John,5925.0,1500.0,3500.0,3000.0,7000.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,2500.0,5000.0,2500.0,5500.0
3,104,David,Cardiology,Dr. Smith,6200.0,3000.0,3200.0,4000.0,8000.0
4,105,Eva,Dermatology,Dr. Rose,5925.0,1000.0,4000.0,NaN,NaN


Department-wise Revenue Analysis

In [33]:
revenue = final_dataset.groupby(
    'Department'
)['BillAmount'].sum()

In [34]:
print(revenue)

Department
Cardiology     19200.0
Dermatology     5925.0
Neurology      15425.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


Doctor-wise Revenue Analysis

In [35]:
doctor_revenue = final_dataset.groupby(
    'Doctor'
)['BillAmount'].sum()

In [36]:
print(doctor_revenue)

Doctor
Dr. John      5925.0
Dr. Kumar     8000.0
Dr. Lee       7500.0
Dr. Rao       9500.0
Dr. Rose      5925.0
Dr. Smith    11200.0
Name: BillAmount, dtype: float64


Save Final Cleaned Dataset

In [37]:
final_dataset.to_csv(
    "Final_Cleaned_Hospital_Data.csv",
    index=False
)

Print Final Dataset

In [38]:
print("Final Cleaned Dataset:")
print(final_dataset)

Final Cleaned Dataset:
   PatientID     Name   Department     Doctor  BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith      5000.0            2000.0   
1        102      Bob    Neurology   Dr. John      5925.0            1500.0   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0            2500.0   
3        104    David   Cardiology  Dr. Smith      6200.0            3000.0   
4        105      Eva  Dermatology   Dr. Rose      5925.0            1000.0   
5        201      NaN   Cardiology  Dr. Kumar      8000.0               NaN   
6        202      NaN    Neurology    Dr. Rao      9500.0               NaN   

   FinalAmount  InsuranceCovered  FinalAmount  
0       3000.0            2000.0       6000.0  
1       3500.0            3000.0       7000.0  
2       5000.0            2500.0       5500.0  
3       3200.0            4000.0       8000.0  
4       4000.0               NaN          NaN  
5          NaN               NaN          NaN  
6       

In [39]:
final_dataset

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,2000.0,3000.0,2000.0,6000.0
1,102,Bob,Neurology,Dr. John,5925.0,1500.0,3500.0,3000.0,7000.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,2500.0,5000.0,2500.0,5500.0
3,104,David,Cardiology,Dr. Smith,6200.0,3000.0,3200.0,4000.0,8000.0
4,105,Eva,Dermatology,Dr. Rose,5925.0,1000.0,4000.0,NaN,NaN
5,201,NaN,Cardiology,Dr. Kumar,8000.0,NaN,NaN,NaN,NaN
6,202,NaN,Neurology,Dr. Rao,9500.0,NaN,NaN,NaN,NaN


In [40]:
display(final_dataset)

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,2000.0,3000.0,2000.0,6000.0
1,102,Bob,Neurology,Dr. John,5925.0,1500.0,3500.0,3000.0,7000.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,2500.0,5000.0,2500.0,5500.0
3,104,David,Cardiology,Dr. Smith,6200.0,3000.0,3200.0,4000.0,8000.0
4,105,Eva,Dermatology,Dr. Rose,5925.0,1000.0,4000.0,NaN,NaN
5,201,NaN,Cardiology,Dr. Kumar,8000.0,NaN,NaN,NaN,NaN
6,202,NaN,Neurology,Dr. Rao,9500.0,NaN,NaN,NaN,NaN


Show Dataset Shape

In [41]:
print("Number of Rows and Columns:")
print(final_dataset.shape)

Number of Rows and Columns:
(7, 9)
